<a href="https://colab.research.google.com/github/routparam12/Python_qns/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Create a model for prediction

In [1]:
import pandas as pd
import sklearn
assert sklearn.__version__ == "1.8.0", sklearn.__version__
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [2]:
#!pip install scikit-learn==1.8.0

In [3]:
pip show scikit-learn


Name: scikit-learn
Version: 1.8.0
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: esda, fastai, hdbscan, imbalanced-learn, libpysal, librosa, mapclassify, mlxtend, pynndescent, pysal, segregation, sentence-transformers, shap, sklearn-compat, sklearn-pandas, spopt, spreg, tsfresh, umap-learn, yellowbrick


In [4]:
df = pd.read_csv('/content/insurance.csv')

In [5]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
13,28,93.4,1.84,11.95,False,Kolkata,freelancer,Low
46,42,83.0,1.57,25.57,True,Kolkata,unemployed,High
82,35,56.0,1.77,12.96,False,Delhi,unemployed,Low
48,36,94.8,1.66,32.69,True,Chennai,unemployed,Medium
24,50,54.2,1.66,18.60,False,Mysore,private_job,Medium


In [6]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [7]:
df_feat = df.copy()

In [8]:
#Feature 1: BMI
df_feat["bmi"]= df_feat["weight"]/(df_feat["height"]/100)**2

In [9]:
# Feature 2 : Age Group
def age_group(age):
  if age < 25:
    return "Young"
  elif age < 45:
    return "Adult"
  elif age < 65:
    return "Middle Age"
  else:
    return "Senior"

In [10]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [11]:
#Feature 3: Lifestyle Risk
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"] > 30:
    return 'high'
  elif row["smoker"] or row["bmi"] > 27:
    return 'medium'
  else :
    return "low"

In [12]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [13]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [14]:
#Feature 4: city Tier
def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3


In [15]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [16]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)


,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
78,14.74,freelancer,279327.976625,Middle Age,high,2,High
88,30.00,government_job,314436.983471,Middle Age,medium,1,Low
93,1.28,student,231994.156318,Young,medium,2,Low
79,30.00,government_job,227235.371466,Adult,high,2,Low
80,50.00,unemployed,343504.607551,Middle Age,medium,2,High


In [17]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [18]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,492274.819198,Senior,medium,2,2.92000,retired
1,301890.172893,Adult,medium,1,34.28000,freelancer
2,211183.819155,Adult,medium,2,36.64000,freelancer
3,455359.001041,Young,high,1,3.34000,student
4,242968.750000,Senior,high,2,3.94000,retired
...,...,...,...,...,...,...
95,214207.472920,Adult,medium,2,19.64000,business_owner
96,479844.830494,Adult,medium,1,34.01000,private_job
97,187654.320988,Middle Age,medium,1,44.86000,freelancer
98,305216.761261,Adult,medium,1,28.30000,business_owner


In [19]:
y

,insurance_premium_category
0,High
1,Low
2,Low
3,Medium
4,High
...,...
95,Low
96,Low
97,Low
98,Low


In [20]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [21]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)


# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])


# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [22]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)


0.85

In [23]:
X_test.sample(5)


,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
33,217910.640496,Senior,medium,1,1.46,retired
32,314958.448753,Middle Age,medium,2,50.00,private_job
51,388279.225728,Middle Age,high,2,28.95,private_job
39,356434.240363,Middle Age,high,1,11.99,unemployed
92,183199.415632,Adult,high,2,30.00,government_job


In [24]:

import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
# Load dataset
data = load_iris()
print(data)
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
# Train a classification tree
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
# Predict and evaluate
predictions = clf.predict(X_test)
print(predictions)

{'data': array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
     

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/join traffic.csv")
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

/tmp/ipykernel_7379/2835671732.py:5: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/join traffic.csv")


(591199, 36)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591199 entries, 0 to 591198
Data columns (total 36 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   panelistId      591199 non-null  int64  
 1   email           591199 non-null  object 
 2   id              591199 non-null  int64  
 3   ipAddress       591199 non-null  object 
 4   latitude        497354 non-null  float64
 5   longitude       497354 non-null  float64
 6   countryCode     591199 non-null  object 
 7   country         591199 non-null  object 
 8   region          0 non-null       float64
 9   organization    587342 non-null  object 
 10  isp             587342 non-null  object 
 11  userType        591199 non-null  object 
 12  fraudScore      591199 non-null  int64  
 13  isVPN           591199 non-null  bool   
 14  isProxy         591199 non-null  bool   
 15  recentAbuse     591199 non-null  bool   
 16  countryAllowed  591199 non-null  bool   
 1

In [3]:
from sklearn.impute import SimpleImputer
import numpy as np

# Separate numerical & categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# Identify numerical columns that are entirely NaN
all_nan_num_cols = [col for col in num_cols if df[col].isnull().all()]

# Exclude these entirely NaN columns from the numerical imputation process
num_cols_for_imputation = [col for col in num_cols if col not in all_nan_num_cols]

# Numerical: mean / median / constant for columns that can be imputed
num_imputer = SimpleImputer(strategy='median')
if num_cols_for_imputation: # Only impute if there are columns to impute
    df[num_cols_for_imputation] = num_imputer.fit_transform(df[num_cols_for_imputation])

# Drop the columns that were entirely NaN (like 'region')
if all_nan_num_cols:
    df = df.drop(columns=all_nan_num_cols)

# Categorical: most frequent or constant
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

# OneHotEncoder for nominal categories
# LabelEncoder only for ordinal or target variable